# Advanced Python Enumerations: 28 Problems, Complete Solutions & Executable Tests

**Topics:** `Enum`, member identity, `name`/`value`, robust lookup, aliases, iteration, `__members__`, hashability, ordering, immutability limits, `auto`, `_missing_`, `@unique`, `IntEnum`, `StrEnum`, `Flag`, `IntFlag`, JSON, dataclasses, state machines, exhaustive coverage, and a transactional capstone.

**Audience:** intermediate-to-advanced Python. **Requirements:** Python **3.11+**, standard library only. Execute the entire notebook top-to-bottom with **Kernel → Restart & Run All**. Problems include complete worked solutions and `assert`-based tests. No network, datasets, credentials, or external packages are needed to run the notebook.

**How to study:** Read each task, try a solution without looking, then inspect the implementation and run its tests. `assert` statements turn the notebook into an executable answer key; a clean run ends with an overall success cell. Each problem uses distinctive identifiers and avoids modifying external files.

**Source alignment:** The uploaded introductory lesson covers basic enum creation, member identity, lookup by name/value, hashing, iteration order, `__members__`, and limits on extending enum classes. The advanced exercises below expand beyond the lesson; they are new exercises, not claims that the source taught advanced APIs.

**Version and correctness notes:** Before Python 3.12, `some_string in SomeEnum` can raise `TypeError`; from 3.12 it checks whether the value occurs. For portable name checks, use `name in MyEnum.__members__`; for member checks, use `isinstance(obj, MyEnum)`. An enum's member/value bindings cannot be reassigned, but a mutable *object held in* `.value` may still change. `StrEnum` and `FlagBoundary` require Python 3.11+. Avoid using arbitrary mutable values as enum constants; creation can become expensive.

**Reference (optional):** [Python enum API](https://docs.python.org/3/library/enum.html) · [Python Enum HOWTO](https://docs.python.org/3/howto/enum.html). The source notebook reports Python 3.6.7, so behavior that varies by Python version is explicitly identified here.

## Environment and assertion helper

We use a tiny context manager for expected failures. Do not suppress unexpected exceptions: those should fail the notebook.

In [1]:
import enum
import json
import sys
from collections import Counter, deque
from contextlib import contextmanager
from dataclasses import dataclass
from typing import Any

assert sys.version_info >= (3, 11), 'Run this notebook with Python 3.11 or newer.'

@contextmanager
def raises(expected_exception):
    try:
        yield
    except expected_exception:
        pass
    else:
        raise AssertionError(f'Expected {expected_exception.__name__} to be raised')

print('Python:', sys.version.split()[0])
print('Standard-library enum exercises ready.')

Python: 3.13.7
Standard-library enum exercises ready.


## Problem 01 — Distinguish member, name, and value

**Your task.** Define an order state with `PENDING`, `RUNNING`, and `DONE`. Implement three **different** validated access paths: member identity, lookup by member name, and lookup by stored value. Reject wrong types and unknown entries rather than silently coerce them. What exceptions should callers see?

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 01

`OrderState01[name]` looks up a **name** and raises `KeyError`; `OrderState01(value)` looks up a **value** and raises `ValueError`. A valid enum member is a singleton, so identity (`is`) is appropriate when matching known members. A string equal to a member's name is still not itself a member. Checking types first provides explicit failure behavior.

In [2]:
class OrderState01(enum.Enum):
    PENDING = 'pending'
    RUNNING = 'running'
    DONE = 'done'

def by_name01(name: str) -> OrderState01:
    if not isinstance(name, str):
        raise TypeError('member name must be a string')
    return OrderState01[name]

def by_value01(value: str) -> OrderState01:
    if not isinstance(value, str):
        raise TypeError('stored value must be a string')
    return OrderState01(value)

def is_done01(candidate: object) -> bool:
    return candidate is OrderState01.DONE

print(OrderState01.DONE.name, OrderState01.DONE.value)
print('Lookup by name:', by_name01('DONE'))
print('Lookup by value:', by_value01('done'))

DONE done
Lookup by name: OrderState01.DONE
Lookup by value: OrderState01.DONE


**Verification — Problem 01.** These assertions cover ordinary inputs and failure modes.

In [3]:
assert by_name01('DONE') is OrderState01.DONE
assert by_value01('done') is OrderState01.DONE
assert is_done01(OrderState01.DONE)
assert not is_done01('done')
assert isinstance(OrderState01.DONE, OrderState01)
with raises(KeyError): by_name01('done')
with raises(ValueError): by_value01('DONE')
with raises(TypeError): by_name01(None)
with raises(TypeError): by_value01(10)
print("Problem 01: PASS")

Problem 01: PASS


**Stretch exercise (not required for the verified solution).** Add a parser that supports *either* spelling but rejects ambiguous strings if name and value collide across different members.

## Problem 02 — Membership that works across Python versions

**Your task.** Implement `has_name`, `has_value`, and `has_member` without depending on Python-version-specific `in Enum` semantics. Include wrong-type checks, a negative case, and at least one cross-enum case.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 02

Check names via the read-only `__members__` mapping (which also includes aliases), values via guarded construction, and actual members via `isinstance`. Avoid `getattr` for untrusted names: methods and class attributes are not necessarily enum members.

In [4]:
class Transport02(enum.Enum):
    CAR = 'car'
    TRAIN = 'train'

class Other02(enum.Enum):
    CAR = 'car'

def has_name02(name: object) -> bool:
    return isinstance(name, str) and name in Transport02.__members__

def has_value02(value: object) -> bool:
    if not isinstance(value, str):
        return False
    try:
        Transport02(value)
    except ValueError:
        return False
    return True

def has_member02(candidate: object) -> bool:
    return isinstance(candidate, Transport02)

print({key: has_name02(key) for key in ['CAR', 'car', 'name']})

{'CAR': True, 'car': False, 'name': False}


**Verification — Problem 02.** These assertions cover ordinary inputs and failure modes.

In [5]:
assert has_name02('CAR') and not has_name02('car')
assert not has_name02(1) and not has_name02('__members__')
assert has_value02('car') and not has_value02('CAR')
assert not has_value02(None)
assert has_member02(Transport02.CAR)
assert not has_member02(Other02.CAR)
assert not has_member02('car')
print("Problem 02: PASS")

Problem 02: PASS


## Problem 03 — Hashable enum keys and frequency analysis

**Your task.** Given a stream of status members, compute a frequency dictionary keyed by members, then return the most frequent member with stable tie-breaking by declaration order. Reject raw strings instead of counting them as valid statuses.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 03

Enum members are hashable and work naturally as dictionary keys even if their associated values are unusual. `Counter` counts members. Iterating the enum in declaration order, then using `max` with the count key, preserves the first member on ties.

In [6]:
class EventKind03(enum.Enum):
    INFO = 'info'
    WARNING = 'warning'
    ERROR = 'error'

def summarize03(events: list[EventKind03]):
    if any(not isinstance(event, EventKind03) for event in events):
        raise TypeError('every event must be an EventKind03 member')
    counts = Counter(events)
    if not events:
        return {}, None
    winner = max(EventKind03, key=lambda member: counts[member])
    return dict(counts), winner

sample03 = [EventKind03.WARNING, EventKind03.INFO, EventKind03.WARNING]
print(summarize03(sample03))

({<EventKind03.WARNING: 'warning'>: 2, <EventKind03.INFO: 'info'>: 1}, <EventKind03.WARNING: 'warning'>)


**Verification — Problem 03.** These assertions cover ordinary inputs and failure modes.

In [7]:
counts03, winner03 = summarize03(sample03)
assert counts03[EventKind03.WARNING] == 2
assert winner03 is EventKind03.WARNING
assert summarize03([]) == ({}, None)
assert summarize03([EventKind03.INFO, EventKind03.WARNING])[1] is EventKind03.INFO
with raises(TypeError): summarize03([EventKind03.ERROR, 'error'])
print("Problem 03: PASS")

Problem 03: PASS


**Stretch exercise (not required for the verified solution).** Adapt the function for streaming iterators without materializing every event.

## Problem 04 — A guarded finite-state machine

**Your task.** Model a job as `PENDING → RUNNING → SUCCEEDED/FAILED`. Only `FAILED → RUNNING` is a permitted retry. Implement a pure transition function that returns the new state and raises an informative error on illegal transitions, including repeated completion.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 04

State enums should represent *states*, not mutable job instances. Keep the transition graph in a separate mapping and use a function to validate edges. Pure functions are easier to test than mutating enum member fields.

In [8]:
class JobState04(enum.Enum):
    PENDING = 'pending'
    RUNNING = 'running'
    SUCCEEDED = 'succeeded'
    FAILED = 'failed'

ALLOWED04 = {
    JobState04.PENDING: frozenset({JobState04.RUNNING}),
    JobState04.RUNNING: frozenset({JobState04.SUCCEEDED, JobState04.FAILED}),
    JobState04.FAILED: frozenset({JobState04.RUNNING}),
    JobState04.SUCCEEDED: frozenset(),
}

def transition04(current: JobState04, target: JobState04) -> JobState04:
    if not isinstance(current, JobState04) or not isinstance(target, JobState04):
        raise TypeError('both states must be JobState04 members')
    if target not in ALLOWED04[current]:
        raise ValueError(f'illegal transition: {current.name} -> {target.name}')
    return target

print(transition04(JobState04.PENDING, JobState04.RUNNING))

JobState04.RUNNING


**Verification — Problem 04.** These assertions cover ordinary inputs and failure modes.

In [9]:
assert transition04(JobState04.PENDING, JobState04.RUNNING) is JobState04.RUNNING
assert transition04(JobState04.FAILED, JobState04.RUNNING) is JobState04.RUNNING
with raises(ValueError): transition04(JobState04.SUCCEEDED, JobState04.RUNNING)
with raises(ValueError): transition04(JobState04.PENDING, JobState04.SUCCEEDED)
with raises(ValueError): transition04(JobState04.RUNNING, JobState04.RUNNING)
with raises(TypeError): transition04('pending', JobState04.RUNNING)
assert set(ALLOWED04) == set(JobState04)
print("Problem 04: PASS")

Problem 04: PASS


**Stretch exercise (not required for the verified solution).** Extend the graph with a `CANCELLED` state and prove terminal states have no outgoing edges.

## Problem 05 — Strict JSON round-trip for API payloads

**Your task.** Encode a task as JSON using the enum **value** as its wire representation, then decode into a typed object. Require exactly `id` and `status`, a non-empty string ID, and a valid status string. Make invalid JSON and unexpected keys fail clearly.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 05

JSON knows strings, not arbitrary enum instances. Serialize `status.value` explicitly and reconstruct using the enum constructor. A frozen dataclass keeps the domain object stable, while strict shape checking avoids accepting malformed data accidentally.

In [10]:
class ApiState05(enum.Enum):
    QUEUED = 'queued'
    ACTIVE = 'active'
    COMPLETE = 'complete'

@dataclass(frozen=True)
class ApiTask05:
    id: str
    status: ApiState05

def encode_task05(task: ApiTask05) -> str:
    if not isinstance(task, ApiTask05) or not isinstance(task.id, str) or not task.id:
        raise TypeError('expected ApiTask05 with non-empty string id')
    if not isinstance(task.status, ApiState05):
        raise TypeError('status must be ApiState05')
    return json.dumps({'id': task.id, 'status': task.status.value}, sort_keys=True)

def decode_task05(payload: str) -> ApiTask05:
    data = json.loads(payload)
    if not isinstance(data, dict) or set(data) != {'id', 'status'}:
        raise ValueError('payload must contain exactly id and status')
    if not isinstance(data['id'], str) or not data['id']:
        raise ValueError('id must be a non-empty string')
    if not isinstance(data['status'], str):
        raise ValueError('status must be a string')
    return ApiTask05(data['id'], ApiState05(data['status']))

wire05 = encode_task05(ApiTask05('T-001', ApiState05.ACTIVE))
print('Wire format:', wire05)
print('Decoded:', decode_task05(wire05))

Wire format: {"id": "T-001", "status": "active"}
Decoded: ApiTask05(id='T-001', status=<ApiState05.ACTIVE: 'active'>)


**Verification — Problem 05.** These assertions cover ordinary inputs and failure modes.

In [11]:
task05 = ApiTask05('T-002', ApiState05.COMPLETE)
assert decode_task05(encode_task05(task05)) == task05
assert json.loads(encode_task05(task05))['status'] == 'complete'
with raises(ValueError): decode_task05('{not json}')
with raises(ValueError): decode_task05('{"id":"X","status":"bogus"}')
with raises(ValueError): decode_task05('{"id":"","status":"queued"}')
with raises(ValueError): decode_task05('{"id":"X","status":"queued","admin":true}')
with raises(ValueError): decode_task05('["X", "queued"]')
print("Problem 05: PASS")

Problem 05: PASS


**Stretch exercise (not required for the verified solution).** Implement versioned payloads, including migration from a legacy status field name.

## Problem 06 — Rich members with structured metadata

**Your task.** Create a shipping tier enum in which each member has a stable string wire code, a display label, and an integer service-level target in days. Look up members by wire code and write a function that compares target days **without** directly ordering enum members.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 06

Override `__new__` to set `_value_` to the wire code and attach descriptive metadata to each singleton. Attached attributes are not inherently immutable, so do not treat them as a security boundary. Keep displayed labels distinct from protocol codes: renaming a UI label should not break persisted values.

In [12]:
class ShippingTier06(enum.Enum):
    ECONOMY = ('economy', 'Economy shipping', 7)
    STANDARD = ('standard', 'Standard shipping', 4)
    EXPRESS = ('express', 'Express shipping', 1)

    def __new__(cls, code: str, label: str, target_days: int):
        obj = object.__new__(cls)
        obj._value_ = code
        obj.label = label
        obj.target_days = target_days
        return obj

def faster06(left: ShippingTier06, right: ShippingTier06) -> ShippingTier06:
    if not isinstance(left, ShippingTier06) or not isinstance(right, ShippingTier06):
        raise TypeError('shipping tiers required')
    return left if left.target_days <= right.target_days else right

for tier in ShippingTier06:
    print(tier.name, tier.value, tier.label, tier.target_days)

ECONOMY economy Economy shipping 7
STANDARD standard Standard shipping 4
EXPRESS express Express shipping 1


**Verification — Problem 06.** These assertions cover ordinary inputs and failure modes.

In [13]:
assert ShippingTier06('express') is ShippingTier06.EXPRESS
assert ShippingTier06.EXPRESS.target_days == 1
assert faster06(ShippingTier06.STANDARD, ShippingTier06.EXPRESS) is ShippingTier06.EXPRESS
with raises(TypeError): _ = ShippingTier06.EXPRESS < ShippingTier06.ECONOMY
with raises(ValueError): ShippingTier06('overnight')
with raises(TypeError): faster06('express', ShippingTier06.STANDARD)
print("Problem 06: PASS")

Problem 06: PASS


**Stretch exercise (not required for the verified solution).** Add a second integer attribute for a published price and determine where that kind of frequently changing data should actually live.

## Problem 07 — Case-insensitive value parsing with `_missing_`

**Your task.** Create a payment status enum accepting inputs like `" PAID "`, `"paid"`, and `"Paid"` by **value**, while rejecting numbers, misspellings, and arbitrary objects. Ensure the normalized parser returns the same canonical member.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 07

Override `_missing_`, which is invoked only when normal value lookup fails. Normalize a string once and compare it with canonical stored values; return `None` for unsupported data so normal Enum lookup raises `ValueError`. Keep output serialization canonical.

In [14]:
class PaymentStatus07(enum.Enum):
    OPEN = 'open'
    PAID = 'paid'
    REFUNDED = 'refunded'

    @classmethod
    def _missing_(cls, value):
        if isinstance(value, str):
            normalized = value.strip().casefold()
            for member in cls:
                if member.value == normalized:
                    return member
        return None

def canonical_payment07(value: str) -> str:
    return PaymentStatus07(value).value

print([canonical_payment07(s) for s in [' PAID ', 'Paid', 'paid']])

['paid', 'paid', 'paid']


**Verification — Problem 07.** These assertions cover ordinary inputs and failure modes.

In [15]:
for sample in ['PAID', ' paid ', 'PaId']:
    assert PaymentStatus07(sample) is PaymentStatus07.PAID
assert canonical_payment07(' Refunded ') == 'refunded'
with raises(ValueError): PaymentStatus07(123)
with raises(ValueError): PaymentStatus07('paid-ish')
with raises(ValueError): PaymentStatus07(None)
print("Problem 07: PASS")

Problem 07: PASS


**Stretch exercise (not required for the verified solution).** Explain why changing the canonical enum `.value` can break API and database compatibility.

## Problem 08 — Aliases, introspection, and uniqueness

**Your task.** Demonstrate an alias, explain why `list(Enum)` and `__members__` yield different sizes, and implement an alias-reporting function. Create a separate unique enum and verify a duplicate declaration is rejected by `@unique`.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 08

Aliases are additional *names* pointing at an existing singleton. Normal iteration skips aliases; `__members__` includes them. `@enum.unique` is the right guard when duplicate values should be prohibited.

In [16]:
class HttpOutcome08(enum.Enum):
    OK = 200
    SUCCESS = 200  # alias of OK
    NOT_FOUND = 404

def aliases08(enum_class: type[enum.Enum]) -> dict[str, str]:
    return {
        name: member.name
        for name, member in enum_class.__members__.items()
        if name != member.name
    }

@enum.unique
class UniqueOutcome08(enum.Enum):
    OK = 200
    NOT_FOUND = 404

print('Canonical members:', list(HttpOutcome08))
print('All names:', list(HttpOutcome08.__members__))
print('Aliases:', aliases08(HttpOutcome08))

Canonical members: [<HttpOutcome08.OK: 200>, <HttpOutcome08.NOT_FOUND: 404>]
All names: ['OK', 'SUCCESS', 'NOT_FOUND']
Aliases: {'SUCCESS': 'OK'}


**Verification — Problem 08.** These assertions cover ordinary inputs and failure modes.

In [17]:
assert HttpOutcome08.SUCCESS is HttpOutcome08.OK
assert HttpOutcome08.SUCCESS.name == 'OK'
assert len(list(HttpOutcome08)) == 2
assert len(HttpOutcome08.__members__) == 3
assert aliases08(HttpOutcome08) == {'SUCCESS': 'OK'}
assert aliases08(UniqueOutcome08) == {}
with raises(ValueError):
    @enum.unique
    class Duplicate08(enum.Enum):
        FIRST = 1
        SECOND = 1
print("Problem 08: PASS")

Problem 08: PASS


**Stretch exercise (not required for the verified solution).** Test how aliases affect JSON round-trips when the original wire format used the alias name.

## Problem 09 — Stable `auto()` generation for wire constants

**Your task.** Create a lowercase-string enum with `auto()` and a custom `_generate_next_value_` so Python's changing default integer-generation rules do not determine your wire representation. Prove all values are unique and definition-ordered.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 09

`_generate_next_value_` is evaluated while the class is being created. A deliberate value-generation strategy is preferable to relying on defaults if a value will be persisted or transmitted. Avoid changing member names after shipping the wire format.

In [18]:
class Action09(enum.Enum):
    @staticmethod
    def _generate_next_value_(name, start, count, last_values):
        return name.lower()

    CREATE = enum.auto()
    UPDATE = enum.auto()
    DELETE = enum.auto()

print([(member.name, member.value) for member in Action09])

[('CREATE', 'create'), ('UPDATE', 'update'), ('DELETE', 'delete')]


**Verification — Problem 09.** These assertions cover ordinary inputs and failure modes.

In [19]:
assert [member.value for member in Action09] == ['create', 'update', 'delete']
assert len({member.value for member in Action09}) == len(Action09)
assert Action09('update') is Action09.UPDATE
assert list(Action09.__members__) == ['CREATE', 'UPDATE', 'DELETE']
with raises(ValueError): Action09('patch')
print("Problem 09: PASS")

Problem 09: PASS


**Stretch exercise (not required for the verified solution).** Create a variant generating prefixed values such as `action:create` and document the migration risk.

## Problem 10 — `IntEnum` compatibility versus type safety

**Your task.** Model a protocol's numeric response codes with `IntEnum`. Show why equality to integers and equality across unrelated `IntEnum` classes can be dangerous. Write a strict converter rejecting `True`, `1.0`, and unsupported numbers.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 10

`IntEnum` deliberately interoperates with integers; Python's `bool` is an `int` subclass. Validate with `type(value) is int` at the trust boundary and use ordinary `Enum` when numeric interoperability is unnecessary.

In [20]:
class ResponseCode10(enum.IntEnum):
    OK = 1
    RETRY = 2
    FATAL = 3

class DifferentCode10(enum.IntEnum):
    OK = 1

def parse_response10(value: object) -> ResponseCode10:
    if type(value) is not int:
        raise TypeError('raw response code must be a real int, not bool or float')
    return ResponseCode10(value)

print('int interoperability:', ResponseCode10.OK == 1)
print('cross-type equality:', ResponseCode10.OK == DifferentCode10.OK)
print('arithmetic result:', ResponseCode10.OK + 1)

int interoperability: True
cross-type equality: True
arithmetic result: 2


**Verification — Problem 10.** These assertions cover ordinary inputs and failure modes.

In [21]:
assert parse_response10(1) is ResponseCode10.OK
assert parse_response10(3) is ResponseCode10.FATAL
assert ResponseCode10.OK == 1
assert ResponseCode10.OK == DifferentCode10.OK
assert type(ResponseCode10.OK + 1) is int
with raises(TypeError): parse_response10(True)
with raises(TypeError): parse_response10(1.0)
with raises(ValueError): parse_response10(999)
print("Problem 10: PASS")

Problem 10: PASS


**Stretch exercise (not required for the verified solution).** Explain when choosing plain `Enum` protects a domain model from accidental integer equality.

## Problem 11 — `StrEnum` and string-operation traps

**Your task.** Define string-compatible API roles using `StrEnum` and `auto()`. Show that string operations produce normal strings rather than new enum members. Write a role parser that refuses input types other than plain `str` and the intended role enum.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 11

`StrEnum` (Python 3.11+) is useful when an API explicitly expects strings. But arithmetic/string methods do not preserve enum types, and equality can cross the type boundary. Validate at input boundaries; reserve ordinary `Enum` for stronger separation.

In [22]:
class ApiRole11(enum.StrEnum):
    READER = enum.auto()
    EDITOR = enum.auto()
    ADMIN = enum.auto()

def parse_role11(raw: object) -> ApiRole11:
    if isinstance(raw, ApiRole11):
        return raw
    if type(raw) is not str:
        raise TypeError('role must be a plain string or ApiRole11')
    return ApiRole11(raw)

upper11 = ApiRole11.READER.upper()
print('role:', ApiRole11.READER, 'uppercase:', upper11, type(upper11))

role: reader uppercase: READER <class 'str'>


**Verification — Problem 11.** These assertions cover ordinary inputs and failure modes.

In [23]:
assert ApiRole11.READER.value == 'reader'
assert ApiRole11.READER == 'reader'
assert type(upper11) is str and upper11 == 'READER'
assert parse_role11('editor') is ApiRole11.EDITOR
assert parse_role11(ApiRole11.ADMIN) is ApiRole11.ADMIN
with raises(TypeError): parse_role11(1)
with raises(ValueError): parse_role11('superuser')
print("Problem 11: PASS")

Problem 11: PASS


**Stretch exercise (not required for the verified solution).** Compare `isinstance(value, str)` with `type(value) is str` for `StrEnum` and subclasses of `str`.

## Problem 12 — Composing permissions with `Flag`

**Your task.** Define read/write/delete permissions as independent flags. Implement `can_perform(granted, required)` so all required bits must be present, including combined requirements. Explain the edge case for a zero-bit requirement.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 12

`Flag` represents sets of independent bits. Require `granted & required == required`; avoid `required in granted` if supporting older Python versions. Here an empty required mask means no permissions are necessary, which is an explicit product decision.

In [24]:
class Permission12(enum.Flag):
    READ = enum.auto()
    WRITE = enum.auto()
    DELETE = enum.auto()

    @classmethod
    def none(cls):
        return cls(0)

def can_perform12(granted: Permission12, required: Permission12) -> bool:
    if not isinstance(granted, Permission12) or not isinstance(required, Permission12):
        raise TypeError('Permission12 masks required')
    return granted & required == required

editor12 = Permission12.READ | Permission12.WRITE
print('editor:', editor12, 'can write?', can_perform12(editor12, Permission12.WRITE))

editor: Permission12.READ|WRITE can write? True


**Verification — Problem 12.** These assertions cover ordinary inputs and failure modes.

In [25]:
assert can_perform12(editor12, Permission12.READ)
assert can_perform12(editor12, Permission12.READ | Permission12.WRITE)
assert not can_perform12(editor12, Permission12.DELETE)
assert not can_perform12(Permission12.READ, Permission12.READ | Permission12.WRITE)
assert can_perform12(Permission12.none(), Permission12.none())
with raises(TypeError): can_perform12(3, Permission12.READ)
assert set(Permission12) == {Permission12.READ, Permission12.WRITE, Permission12.DELETE}
print("Problem 12: PASS")

Problem 12: PASS


**Stretch exercise (not required for the verified solution).** Define a policy where zero required bits is rejected, and test it separately.

## Problem 13 — Parse untrusted `IntFlag` bitmasks safely

**Your task.** Allow an integer bitmask from an API but reject negatives, booleans, non-integers, and any unknown bits—even when `IntFlag` itself might accept them. Return a typed mask and show a round-trip.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 13

`IntFlag` defaults can permit unexpected bits depending on boundary configuration. Validate against an explicit union of recognized bits **before** converting. The bitwise complement of that union identifies unknown bits.

In [26]:
class Capability13(enum.IntFlag):
    VIEW = 1
    CHANGE = 2
    EXPORT = 4

KNOWN_BITS13 = int(Capability13.VIEW | Capability13.CHANGE | Capability13.EXPORT)

def parse_capabilities13(raw: object) -> Capability13:
    if type(raw) is not int:
        raise TypeError('mask must be a plain integer')
    if raw < 0 or raw & ~KNOWN_BITS13:
        raise ValueError('negative or unrecognized capability bits')
    return Capability13(raw)

print('all bits:', KNOWN_BITS13, 'parsed:', parse_capabilities13(5))

all bits: 7 parsed: 5


**Verification — Problem 13.** These assertions cover ordinary inputs and failure modes.

In [27]:
assert parse_capabilities13(0) == Capability13(0)
assert parse_capabilities13(5) == Capability13.VIEW | Capability13.EXPORT
assert int(parse_capabilities13(7)) == 7
for bad in [-1, 8, 9, 1024]:
    with raises(ValueError): parse_capabilities13(bad)
for bad in [True, 3.0, '3', None]:
    with raises(TypeError): parse_capabilities13(bad)
print("Problem 13: PASS")

Problem 13: PASS


**Stretch exercise (not required for the verified solution).** Contrast forward compatibility (preserving unknown bits) with fail-closed authorization checks.

## Problem 14 — Unknown flag boundaries are a policy choice

**Your task.** Demonstrate `STRICT` and `CONFORM` flag boundaries with a known one-bit flag and an unknown high bit. Prove one rejects invalid data while the other drops unknown bits. Describe why silently dropping bits may be inappropriate for security decisions.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 14

Python 3.11+ lets each `Flag` choose a boundary policy. At a trust boundary, `STRICT` is a useful default; `CONFORM` is suitable only when ignoring unknown bits is an explicit requirement.

In [28]:
class StrictFeatures14(enum.Flag, boundary=enum.STRICT):
    A = 1
    B = 2

class ConformingFeatures14(enum.Flag, boundary=enum.CONFORM):
    A = 1
    B = 2

print('conformed:', ConformingFeatures14(1 | 8))
with raises(ValueError):
    StrictFeatures14(1 | 8)

conformed: ConformingFeatures14.A


**Verification — Problem 14.** These assertions cover ordinary inputs and failure modes.

In [29]:
assert StrictFeatures14(1) is StrictFeatures14.A
assert ConformingFeatures14(1 | 8) is ConformingFeatures14.A
assert ConformingFeatures14(8) == ConformingFeatures14(0)
with raises(ValueError): StrictFeatures14(8)
print("Problem 14: PASS")

Problem 14: PASS


**Stretch exercise (not required for the verified solution).** Explore `EJECT` and `KEEP` in the official Python docs; explain how they change the type and handling of unknown bits.

## Problem 15 — Parse a strict configuration mapping

**Your task.** Convert a dictionary such as `{"LOG_LEVEL": "warning", "MODE": "safe"}` into enum-backed settings. Require exactly those two keys, plain strings, case-insensitive values, and specific error messages. Do not trust arbitrary properties on the enum class as member names.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 15

A configuration schema is explicit about its keys and input types. Implement a small parser using constructors after normalization, making invalid configurations fail at startup rather than later during execution.

In [30]:
class LogLevel15(enum.Enum):
    DEBUG = 'debug'
    INFO = 'info'
    WARNING = 'warning'

class Mode15(enum.Enum):
    SAFE = 'safe'
    FAST = 'fast'

@dataclass(frozen=True)
class Settings15:
    level: LogLevel15
    mode: Mode15

def parse_settings15(raw: object) -> Settings15:
    if not isinstance(raw, dict) or set(raw) != {'LOG_LEVEL', 'MODE'}:
        raise ValueError('expected exactly LOG_LEVEL and MODE')
    level, mode = raw['LOG_LEVEL'], raw['MODE']
    if type(level) is not str or type(mode) is not str:
        raise TypeError('configuration values must be plain strings')
    return Settings15(LogLevel15(level.strip().lower()), Mode15(mode.strip().lower()))

settings15 = parse_settings15({'LOG_LEVEL': ' Warning ', 'MODE': 'SAFE'})
print(settings15)

Settings15(level=<LogLevel15.WARNING: 'warning'>, mode=<Mode15.SAFE: 'safe'>)


**Verification — Problem 15.** These assertions cover ordinary inputs and failure modes.

In [31]:
assert settings15 == Settings15(LogLevel15.WARNING, Mode15.SAFE)
with raises(ValueError): parse_settings15({'LOG_LEVEL': 'info'})
with raises(ValueError): parse_settings15({'LOG_LEVEL': 'info', 'MODE': 'safe', 'DEBUG': 'yes'})
with raises(ValueError): parse_settings15({'LOG_LEVEL': 'verbose', 'MODE': 'safe'})
with raises(TypeError): parse_settings15({'LOG_LEVEL': 1, 'MODE': 'safe'})
with raises(ValueError): parse_settings15(None)
print("Problem 15: PASS")

Problem 15: PASS


**Stretch exercise (not required for the verified solution).** Provide defaults *only* for intentionally optional keys, and explain how that changes failure behavior.

## Problem 16 — Exhaustive enum-driven dispatch

**Your task.** Replace an error-prone string `if/elif` chain with a mapping from enum operations to functions. Ensure the dispatch table covers **every** defined operation and no foreign keys. Protect integer division by zero and reject invalid operation input types.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 16

Using enum members as keys catches spelling errors early. Validate exact coverage (`set(table) == set(Enum)`) once, then perform an O(1)-average dictionary dispatch. Keep operation-specific domain errors visible.

In [32]:
class ArithmeticOp16(enum.Enum):
    ADD = 'add'
    SUB = 'sub'
    MUL = 'mul'
    DIV = 'div'

def safe_div16(a: float, b: float) -> float:
    if b == 0:
        raise ZeroDivisionError('division by zero')
    return a / b

OPERATIONS16 = {
    ArithmeticOp16.ADD: lambda a, b: a + b,
    ArithmeticOp16.SUB: lambda a, b: a - b,
    ArithmeticOp16.MUL: lambda a, b: a * b,
    ArithmeticOp16.DIV: safe_div16,
}
assert set(OPERATIONS16) == set(ArithmeticOp16)

def evaluate16(op: ArithmeticOp16, a: float, b: float) -> float:
    if not isinstance(op, ArithmeticOp16):
        raise TypeError('operation must be an ArithmeticOp16 member')
    return OPERATIONS16[op](a, b)

print('7 * 6 =', evaluate16(ArithmeticOp16.MUL, 7, 6))

7 * 6 = 42


**Verification — Problem 16.** These assertions cover ordinary inputs and failure modes.

In [33]:
assert evaluate16(ArithmeticOp16.ADD, 3, 4) == 7
assert evaluate16(ArithmeticOp16.SUB, 3, 4) == -1
assert evaluate16(ArithmeticOp16.MUL, 7, 6) == 42
assert evaluate16(ArithmeticOp16.DIV, 8, 2) == 4
with raises(ZeroDivisionError): evaluate16(ArithmeticOp16.DIV, 1, 0)
with raises(TypeError): evaluate16('add', 3, 4)
assert set(OPERATIONS16) == set(ArithmeticOp16)
print("Problem 16: PASS")

Problem 16: PASS


**Stretch exercise (not required for the verified solution).** Compare explicit dictionary dispatch to Python 3.10+ structural pattern matching.

## Problem 17 — Ordering without pretending `Enum` is ordered

**Your task.** Given severity levels `LOW`, `MEDIUM`, `HIGH`, and `CRITICAL`, implement a typed `at_least` predicate and sort a list from highest to lowest priority. Do not use relational operators directly on ordinary enum members.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 17

Ordinary `Enum` has identity/equality semantics but deliberately lacks `<` and `>`. Place the business-defined rank in a separate mapping. This also allows rank to change without altering wire values.

In [34]:
class Severity17(enum.Enum):
    LOW = 'low'
    MEDIUM = 'medium'
    HIGH = 'high'
    CRITICAL = 'critical'

RANK17 = {
    Severity17.LOW: 10,
    Severity17.MEDIUM: 20,
    Severity17.HIGH: 30,
    Severity17.CRITICAL: 40,
}
assert set(RANK17) == set(Severity17)

def at_least17(current: Severity17, threshold: Severity17) -> bool:
    if not isinstance(current, Severity17) or not isinstance(threshold, Severity17):
        raise TypeError('severity enum members required')
    return RANK17[current] >= RANK17[threshold]

def highest_first17(levels: list[Severity17]) -> list[Severity17]:
    if any(not isinstance(level, Severity17) for level in levels):
        raise TypeError('invalid severity in list')
    return sorted(levels, key=RANK17.__getitem__, reverse=True)

print(highest_first17([Severity17.LOW, Severity17.CRITICAL, Severity17.MEDIUM]))

[<Severity17.CRITICAL: 'critical'>, <Severity17.MEDIUM: 'medium'>, <Severity17.LOW: 'low'>]


**Verification — Problem 17.** These assertions cover ordinary inputs and failure modes.

In [35]:
assert at_least17(Severity17.HIGH, Severity17.MEDIUM)
assert not at_least17(Severity17.LOW, Severity17.HIGH)
assert highest_first17([Severity17.LOW, Severity17.HIGH]) == [Severity17.HIGH, Severity17.LOW]
with raises(TypeError): _ = Severity17.HIGH > Severity17.MEDIUM
with raises(TypeError): at_least17('high', Severity17.LOW)
assert len(RANK17) == len(Severity17)
print("Problem 17: PASS")

Problem 17: PASS


**Stretch exercise (not required for the verified solution).** Make a stable secondary ordering for two events with the same severity.

## Problem 18 — Enum immutability is not deep immutability

**Your task.** Demonstrate that rebinding `.value` fails yet mutating a list stored inside `.value` can succeed. Then redesign the values as immutable tuples and show mutation attempts fail. Explain why mutable enum values should generally be avoided.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 18

Enums protect member definitions, not the internals of an arbitrary mutable object held in `.value`. A list inside an enum singleton is global shared state. Immutable tuple payloads and a separate domain object are safer patterns.

In [36]:
class MutablePayload18(enum.Enum):
    OPTIONS = ['a', 'b']

class SafePayload18(enum.Enum):
    OPTIONS = ('a', 'b')

before18 = list(MutablePayload18.OPTIONS.value)
MutablePayload18.OPTIONS.value.append('c')  # demonstration ONLY
print('Mutable member value changed:', MutablePayload18.OPTIONS.value)
print('Immutable member value:', SafePayload18.OPTIONS.value)

Mutable member value changed: ['a', 'b', 'c']
Immutable member value: ('a', 'b')


**Verification — Problem 18.** These assertions cover ordinary inputs and failure modes.

In [37]:
assert before18 == ['a', 'b']
assert MutablePayload18.OPTIONS.value == ['a', 'b', 'c']
assert SafePayload18.OPTIONS.value == ('a', 'b')
with raises(AttributeError): MutablePayload18.OPTIONS.value = ['x']
with raises(AttributeError): SafePayload18.OPTIONS.value.append('c')
with raises(TypeError): SafePayload18.OPTIONS.value[0] = 'x'
# Restore this demo's shared mutable value so notebook reruns stay interpretable.
MutablePayload18.OPTIONS.value.pop()
assert MutablePayload18.OPTIONS.value == ['a', 'b']
print("Problem 18: PASS")

Problem 18: PASS


**Stretch exercise (not required for the verified solution).** Replace tuple payloads with frozen dataclasses for named immutable fields; note nested mutability concerns.

## Problem 19 — Legal and illegal enum inheritance

**Your task.** Build an empty enum base that supplies a description method, then inherit it and add members. Show that trying to subclass an enum that already has members raises `TypeError`. Keep the failed class declaration inside an expected-exception test.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 19

An enum with members cannot be extended by subclassing because that would undermine member identity and type guarantees. Reuse methods through an empty enum base class or a mixin instead.

In [38]:
class DescribedBase19(enum.Enum):
    def describe(self) -> str:
        return f'{type(self).__name__}.{self.name}={self.value!r}'

class Planet19(DescribedBase19):
    EARTH = 3
    MARS = 4

print(Planet19.EARTH.describe())

Planet19.EARTH=3


**Verification — Problem 19.** These assertions cover ordinary inputs and failure modes.

In [39]:
assert Planet19.EARTH.describe() == 'Planet19.EARTH=3'
assert list(Planet19) == [Planet19.EARTH, Planet19.MARS]
with raises(TypeError):
    class IllegalExtension19(Planet19):
        JUPITER = 5
assert 'JUPITER' not in Planet19.__members__
print("Problem 19: PASS")

Problem 19: PASS


**Stretch exercise (not required for the verified solution).** Create a string-formatting mixin and reuse it across two distinct enums, without sharing actual members.

## Problem 20 — Dataclass domain validation and frozen objects

**Your task.** Represent a typed support ticket with a string ID, enum priority, and enum state. Reject invalid fields in `__post_init__`. Implement a pure update function using `dataclasses.replace`, leaving the original ticket unchanged.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 20

A frozen dataclass freezes the ticket **instance**, while enum members represent allowable constants. Validate types at construction time and return new instances for changes: this avoids mutable singleton anti-patterns.

In [40]:
from dataclasses import FrozenInstanceError, replace

class TicketPriority20(enum.Enum):
    NORMAL = 'normal'
    URGENT = 'urgent'

class TicketState20(enum.Enum):
    OPEN = 'open'
    CLOSED = 'closed'

@dataclass(frozen=True)
class Ticket20:
    ticket_id: str
    priority: TicketPriority20
    state: TicketState20

    def __post_init__(self):
        if not isinstance(self.ticket_id, str) or not self.ticket_id.strip():
            raise ValueError('ticket_id must be non-empty')
        if not isinstance(self.priority, TicketPriority20):
            raise TypeError('priority must be TicketPriority20')
        if not isinstance(self.state, TicketState20):
            raise TypeError('state must be TicketState20')

def close_ticket20(ticket: Ticket20) -> Ticket20:
    if ticket.state is not TicketState20.OPEN:
        raise ValueError('ticket is not open')
    return replace(ticket, state=TicketState20.CLOSED)

ticket20 = Ticket20('SUP-9', TicketPriority20.URGENT, TicketState20.OPEN)
closed20 = close_ticket20(ticket20)
print(ticket20, '->', closed20)

Ticket20(ticket_id='SUP-9', priority=<TicketPriority20.URGENT: 'urgent'>, state=<TicketState20.OPEN: 'open'>) -> Ticket20(ticket_id='SUP-9', priority=<TicketPriority20.URGENT: 'urgent'>, state=<TicketState20.CLOSED: 'closed'>)


**Verification — Problem 20.** These assertions cover ordinary inputs and failure modes.

In [41]:
assert ticket20.state is TicketState20.OPEN
assert closed20.state is TicketState20.CLOSED
assert closed20.ticket_id == ticket20.ticket_id
with raises(ValueError): close_ticket20(closed20)
with raises(TypeError): Ticket20('X', 'urgent', TicketState20.OPEN)
with raises(ValueError): Ticket20('', TicketPriority20.NORMAL, TicketState20.OPEN)
with raises(FrozenInstanceError): ticket20.state = TicketState20.CLOSED
print("Problem 20: PASS")

Problem 20: PASS


**Stretch exercise (not required for the verified solution).** Add JSON serialization using values and prove the original ticket remains unchanged when serialization fails.

## Problem 21 — Command-line parsing without invoking a shell

**Your task.** Use `argparse` to parse `--format json|csv` into an enum. Test valid and invalid arguments without reading the notebook process's `sys.argv`, and capture failures without printing noisy usage text.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 21

`argparse` should receive a supplied argument list in tests. `choices` validates the enum's wire values; convert the selected string immediately into an enum member. Subclassing `ArgumentParser.error` can turn invalid CLI input into an ordinary exception in notebook tests.

In [42]:
import argparse

class ExportFormat21(enum.Enum):
    JSON = 'json'
    CSV = 'csv'

class QuietParser21(argparse.ArgumentParser):
    def error(self, message):
        raise ValueError(message)

def parse_cli21(args: list[str]) -> ExportFormat21:
    parser = QuietParser21(add_help=False)
    parser.add_argument('--format', required=True,
                        choices=[member.value for member in ExportFormat21])
    namespace = parser.parse_args(args)
    return ExportFormat21(namespace.format)

print(parse_cli21(['--format', 'json']))

ExportFormat21.JSON


**Verification — Problem 21.** These assertions cover ordinary inputs and failure modes.

In [43]:
assert parse_cli21(['--format', 'json']) is ExportFormat21.JSON
assert parse_cli21(['--format', 'csv']) is ExportFormat21.CSV
with raises(ValueError): parse_cli21(['--format', 'xml'])
with raises(ValueError): parse_cli21([])
with raises(ValueError): parse_cli21(['--unknown', 'x'])
print("Problem 21: PASS")

Problem 21: PASS


**Stretch exercise (not required for the verified solution).** Add another typed argument and ensure error messages identify the invalid flag or value.

## Problem 22 — Migrate legacy wire values explicitly

**Your task.** An API used `"in_progress"` and now uses `"running"`, while old clients still send the former. Accept both on input but **always** emit the canonical new value on output. Reject unknown values and non-string input.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 22

Avoid creating an enum alias with a distinct stored value—that is not how aliases work. Use a migration mapping at the boundary, then create the canonical enum and only serialize `.value`. Keep the legacy mapping out of internal domain logic.

In [44]:
class Lifecycle22(enum.Enum):
    PENDING = 'pending'
    RUNNING = 'running'
    DONE = 'done'

LEGACY_VALUES22 = {'in_progress': 'running', 'finished': 'done'}

def decode_lifecycle22(raw: object) -> Lifecycle22:
    if type(raw) is not str:
        raise TypeError('lifecycle must be a plain string')
    normalized = LEGACY_VALUES22.get(raw, raw)
    return Lifecycle22(normalized)

def encode_lifecycle22(state: Lifecycle22) -> str:
    if not isinstance(state, Lifecycle22):
        raise TypeError('lifecycle member required')
    return state.value

print('Legacy decoded:', decode_lifecycle22('in_progress'))
print('Canonical encoded:', encode_lifecycle22(decode_lifecycle22('in_progress')))

Legacy decoded: Lifecycle22.RUNNING
Canonical encoded: running


**Verification — Problem 22.** These assertions cover ordinary inputs and failure modes.

In [45]:
assert decode_lifecycle22('in_progress') is Lifecycle22.RUNNING
assert decode_lifecycle22('finished') is Lifecycle22.DONE
assert encode_lifecycle22(decode_lifecycle22('finished')) == 'done'
assert encode_lifecycle22(Lifecycle22.PENDING) == 'pending'
with raises(ValueError): decode_lifecycle22('complete-ish')
with raises(TypeError): decode_lifecycle22(None)
with raises(TypeError): encode_lifecycle22('done')
print("Problem 22: PASS")

Problem 22: PASS


**Stretch exercise (not required for the verified solution).** Add a deprecation counter that records each legacy value without polluting business logic.

## Problem 23 — Verify a state machine using graph traversal

**Your task.** Given enum states and a transition graph, verify exhaustive graph keys and reachability from `NEW`. Confirm that `DONE` is terminal, `FAILED` can reach `DONE` through a retry, and every state is reachable. Implement BFS; do not hard-code a chain of special cases.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 23

Treat enum states as graph vertices. A queue-based breadth-first search visits every reachable member once and handles cycles. Validate graph coverage *before* traversing; missing keys should be a test failure rather than a hidden assumption.

In [46]:
class Workflow23(enum.Enum):
    NEW = 'new'
    WORKING = 'working'
    FAILED = 'failed'
    DONE = 'done'

GRAPH23 = {
    Workflow23.NEW: frozenset({Workflow23.WORKING}),
    Workflow23.WORKING: frozenset({Workflow23.FAILED, Workflow23.DONE}),
    Workflow23.FAILED: frozenset({Workflow23.WORKING}),
    Workflow23.DONE: frozenset(),
}

def reachable23(start: Workflow23) -> frozenset[Workflow23]:
    if not isinstance(start, Workflow23):
        raise TypeError('start must be Workflow23')
    if set(GRAPH23) != set(Workflow23):
        raise AssertionError('transition graph is not exhaustive')
    if any(not set(neighbors) <= set(Workflow23) for neighbors in GRAPH23.values()):
        raise AssertionError('graph contains foreign states')
    seen = {start}
    queue = deque([start])
    while queue:
        current = queue.popleft()
        for neighbor in GRAPH23[current]:
            if neighbor not in seen:
                seen.add(neighbor)
                queue.append(neighbor)
    return frozenset(seen)

print('Reachable from NEW:', sorted(s.name for s in reachable23(Workflow23.NEW)))

Reachable from NEW: ['DONE', 'FAILED', 'NEW', 'WORKING']


**Verification — Problem 23.** These assertions cover ordinary inputs and failure modes.

In [47]:
assert reachable23(Workflow23.NEW) == frozenset(Workflow23)
assert Workflow23.DONE in reachable23(Workflow23.FAILED)
assert reachable23(Workflow23.DONE) == frozenset({Workflow23.DONE})
assert not GRAPH23[Workflow23.DONE]
with raises(TypeError): reachable23('new')
print("Problem 23: PASS")

Problem 23: PASS


**Stretch exercise (not required for the verified solution).** Compute shortest transition paths and disallow particular edges at runtime by role.

## Problem 24 — Safe generic wire-code registry

**Your task.** Build a registry for multiple enums with a unique type tag (`"state"`, `"priority"`) and unique wire values per enum. Implement `encode(member)` and `decode(tag, value)` without dynamic attribute access or evaluating untrusted input.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 24

The registry is an explicit whitelist, not a reflection-based import system. Validate enum uniqueness (including aliases), reject unknown tags, and look up values only within the registered class. Keep member namespaces independent across enums.

In [48]:
@enum.unique
class WireState24(enum.Enum):
    OPEN = 'open'
    CLOSED = 'closed'

@enum.unique
class WirePriority24(enum.Enum):
    LOW = 'low'
    HIGH = 'high'

REGISTRY24 = {'state': WireState24, 'priority': WirePriority24}
REVERSE24 = {enum_type: tag for tag, enum_type in REGISTRY24.items()}
assert len(REVERSE24) == len(REGISTRY24)

def encode_enum24(member: enum.Enum) -> dict[str, str]:
    enum_type = type(member)
    if enum_type not in REVERSE24:
        raise TypeError('unregistered enum member')
    if type(member.value) is not str:
        raise TypeError('wire values must be plain strings')
    return {'type': REVERSE24[enum_type], 'value': member.value}

def decode_enum24(payload: object) -> enum.Enum:
    if not isinstance(payload, dict) or set(payload) != {'type', 'value'}:
        raise ValueError('expected exactly type and value')
    tag, value = payload['type'], payload['value']
    if type(tag) is not str or tag not in REGISTRY24:
        raise ValueError('unknown enum type tag')
    if type(value) is not str:
        raise TypeError('wire value must be a plain string')
    return REGISTRY24[tag](value)

print(encode_enum24(WirePriority24.HIGH))

{'type': 'priority', 'value': 'high'}


**Verification — Problem 24.** These assertions cover ordinary inputs and failure modes.

In [49]:
for member in list(WireState24) + list(WirePriority24):
    assert decode_enum24(encode_enum24(member)) is member
with raises(ValueError): decode_enum24({'type': 'missing', 'value': 'open'})
with raises(ValueError): decode_enum24({'type': 'state', 'value': 'high'})
with raises(ValueError): decode_enum24({'type': 'state'})
with raises(TypeError): decode_enum24({'type': 'state', 'value': 1})
with raises(TypeError): encode_enum24(JobState04.PENDING)
print("Problem 24: PASS")

Problem 24: PASS


**Stretch exercise (not required for the verified solution).** Introduce explicit schema versioning for each registered tag and migration functions.

## Problem 25 — Exhaustive bitmask property tests without dependencies

**Your task.** For every possible three-bit permission mask and every required mask, prove that `can_perform` agrees with the mathematical bit-subset rule. Test unknown-bit rejection for a broad range of invalid masks.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 25

Exhaustive testing is realistic for small bit domains: three independent bits give only eight masks, so there are 64 ordered pairs. Use nested loops and invariants instead of checking only hand-selected examples.

In [50]:
def check_all_masks25() -> int:
    cases = 0
    for granted_raw in range(8):
        granted = Permission12(granted_raw)
        for required_raw in range(8):
            required = Permission12(required_raw)
            expected = (granted_raw & required_raw) == required_raw
            actual = can_perform12(granted, required)
            assert actual == expected, (granted_raw, required_raw)
            cases += 1
    return cases

checked25 = check_all_masks25()
print(f'Exhaustively verified {checked25} permission combinations')

Exhaustively verified 64 permission combinations


**Verification — Problem 25.** These assertions cover ordinary inputs and failure modes.

In [51]:
assert checked25 == 64
for unknown_mask in range(8, 128):
    with raises(ValueError): parse_capabilities13(unknown_mask)
for known_mask in range(8):
    assert int(parse_capabilities13(known_mask)) == known_mask
print("Problem 25: PASS")

Problem 25: PASS


**Stretch exercise (not required for the verified solution).** Extend to five independent flags, and compare exhaustive testing against randomized testing.

## Problem 26 — Read-only schema introspection and coverage

**Your task.** Build a serializable list of enum schema records (`name`, `value`, `alias_of`). Include aliases, preserve definition order, and reject non-JSON-compatible stored values rather than silently stringifying objects.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 26

`__members__` is a read-only ordered mapping including aliases. Constrain supported values explicitly for schema export; `repr()` of arbitrary Python objects is not a stable JSON protocol. Avoid assuming iteration lists every declared name.

In [52]:
class ApiResult26(enum.Enum):
    SUCCESS = 0
    OK = 0
    BAD_REQUEST = 400
    FAILURE = 500

def export_schema26(enum_class: type[enum.Enum]) -> list[dict[str, object]]:
    if not isinstance(enum_class, enum.EnumType):
        raise TypeError('enum class required')
    records = []
    for declared_name, member in enum_class.__members__.items():
        if type(member.value) not in (str, int, float, bool, type(None)):
            raise TypeError('schema values must be simple JSON scalars')
        records.append({
            'name': declared_name,
            'value': member.value,
            'alias_of': None if declared_name == member.name else member.name,
        })
    return records

schema26 = export_schema26(ApiResult26)
print(json.dumps(schema26, indent=2))

[
  {
    "name": "SUCCESS",
    "value": 0,
    "alias_of": null
  },
  {
    "name": "OK",
    "value": 0,
    "alias_of": "SUCCESS"
  },
  {
    "name": "BAD_REQUEST",
    "value": 400,
    "alias_of": null
  },
  {
    "name": "FAILURE",
    "value": 500,
    "alias_of": null
  }
]


**Verification — Problem 26.** These assertions cover ordinary inputs and failure modes.

In [53]:
assert [row['name'] for row in schema26] == ['SUCCESS', 'OK', 'BAD_REQUEST', 'FAILURE']
assert schema26[1]['alias_of'] == 'SUCCESS'
assert schema26[0]['alias_of'] is None
assert len(schema26) == len(ApiResult26.__members__)
assert len(list(ApiResult26)) == 3
with raises(TypeError): export_schema26('not an enum class')
with raises(TypeError): export_schema26(MutablePayload18)
print("Problem 26: PASS")

Problem 26: PASS


**Stretch exercise (not required for the verified solution).** Add an enum checksum for detecting accidental wire-schema changes in CI.

## Problem 27 — Type-safe cross-enum equality and identity

**Your task.** Show that two ordinary enums with equal values are not equal, aliases of the **same** enum are identical, and `IntEnum` can cross type boundaries. Write a helper that compares only members of the same enum class and never coalesces unrelated enums.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 27

Identity and type checks make intent clear. `Enum` isolates unrelated domains even when `.value` matches; `IntEnum` intentionally abandons some isolation. Enforce `type(a) is type(b)` for a generic same-enum comparison.

In [54]:
class RedTeam27(enum.Enum):
    ACTIVE = 1
    ENABLED = 1

class BlueTeam27(enum.Enum):
    ACTIVE = 1

def same_member27(a: object, b: object) -> bool:
    return (isinstance(a, enum.Enum)
            and isinstance(b, enum.Enum)
            and type(a) is type(b)
            and a is b)

print('alias identity:', RedTeam27.ACTIVE is RedTeam27.ENABLED)
print('cross-enum equality:', RedTeam27.ACTIVE == BlueTeam27.ACTIVE)

alias identity: True
cross-enum equality: False


**Verification — Problem 27.** These assertions cover ordinary inputs and failure modes.

In [55]:
assert same_member27(RedTeam27.ACTIVE, RedTeam27.ENABLED)
assert not same_member27(RedTeam27.ACTIVE, BlueTeam27.ACTIVE)
assert not same_member27(RedTeam27.ACTIVE, 1)
assert RedTeam27.ACTIVE != BlueTeam27.ACTIVE
assert RedTeam27.ACTIVE.value == BlueTeam27.ACTIVE.value
assert RedTeam27.ACTIVE.name == 'ACTIVE' 
print("Problem 27: PASS")

Problem 27: PASS


**Stretch exercise (not required for the verified solution).** Write a type checker-friendly version using `TypeVar` and explain the remaining runtime checks.

## Problem 28 — Capstone: versioned, permissioned order workflow

**Your task.** Implement an immutable order record with state, revision, and audit entries. Commands must validate a typed target, an exact expected revision (optimistic concurrency), caller permissions, and allowed transitions. Return a new record on success; on any failure, preserve the old one. Create/round-trip a strictly validated JSON snapshot and test stale revisions, forbidden changes, and malformed payloads.

**Try it first:** write your own implementation before expanding the solution below. Specify what must happen for invalid inputs and test at least one boundary case.

### Worked solution 28

Compose the prior techniques rather than putting per-order state inside enum singletons: `Enum` defines states, `Flag` defines rights, frozen dataclasses define records and audit events, a transition graph defines domain rules, an exact revision check avoids lost updates, and JSON uses explicit stable wire values. This is a deterministic in-memory model, not a database transaction or production authorization system.

In [56]:
class OrderPhase28(enum.Enum):
    NEW = 'new'
    PAID = 'paid'
    SHIPPED = 'shipped'
    CANCELLED = 'cancelled'

class OrderRight28(enum.Flag):
    PAY = enum.auto()
    SHIP = enum.auto()
    CANCEL = enum.auto()

TRANSITIONS28 = {
    OrderPhase28.NEW: {
        OrderPhase28.PAID: OrderRight28.PAY,
        OrderPhase28.CANCELLED: OrderRight28.CANCEL,
    },
    OrderPhase28.PAID: {
        OrderPhase28.SHIPPED: OrderRight28.SHIP,
        OrderPhase28.CANCELLED: OrderRight28.CANCEL,
    },
    OrderPhase28.SHIPPED: {},
    OrderPhase28.CANCELLED: {},
}
assert set(TRANSITIONS28) == set(OrderPhase28)

@dataclass(frozen=True)
class AuditEntry28:
    from_phase: OrderPhase28
    to_phase: OrderPhase28
    actor: str
    revision: int

@dataclass(frozen=True)
class OrderRecord28:
    order_id: str
    phase: OrderPhase28
    revision: int = 0
    audit: tuple[AuditEntry28, ...] = ()

    def __post_init__(self):
        if type(self.order_id) is not str or not self.order_id:
            raise ValueError('order_id must be a non-empty string')
        if not isinstance(self.phase, OrderPhase28):
            raise TypeError('phase must be OrderPhase28')
        if type(self.revision) is not int or self.revision < 0:
            raise ValueError('revision must be a nonnegative plain int')
        if type(self.audit) is not tuple or any(type(entry) is not AuditEntry28 for entry in self.audit):
            raise TypeError('audit must be a tuple of AuditEntry28')
        if len(self.audit) != self.revision:
            raise ValueError('one audit entry required per revision')
        for position, entry in enumerate(self.audit, start=1):
            if entry.revision != position:
                raise ValueError('audit revision sequence is invalid')
            if position == 1 and entry.from_phase is not OrderPhase28.NEW:
                raise ValueError('first transition must originate in NEW')
            if position > 1 and entry.from_phase is not self.audit[position - 2].to_phase:
                raise ValueError('audit phase sequence is invalid')
            if entry.to_phase not in TRANSITIONS28.get(entry.from_phase, {}):
                raise ValueError('audit contains illegal transition')
            if type(entry.actor) is not str or not entry.actor:
                raise ValueError('audit actor must be a non-empty string')
        if self.audit and self.audit[-1].to_phase is not self.phase:
            raise ValueError('phase must match latest audit transition')
        if not self.audit and self.phase is not OrderPhase28.NEW:
            raise ValueError('new records must start in NEW')


def change_order28(order: OrderRecord28, *, target: OrderPhase28,
                   rights: OrderRight28, actor: str,
                   expected_revision: int) -> OrderRecord28:
    if not isinstance(order, OrderRecord28):
        raise TypeError('order must be OrderRecord28')
    if not isinstance(target, OrderPhase28) or not isinstance(rights, OrderRight28):
        raise TypeError('enum target and rights required')
    if type(actor) is not str or not actor:
        raise ValueError('actor must be a non-empty string')
    if type(expected_revision) is not int or expected_revision < 0:
        raise ValueError('expected_revision must be a nonnegative plain int')
    if expected_revision != order.revision:
        raise RuntimeError('stale order revision')
    required = TRANSITIONS28[order.phase].get(target)
    if required is None:
        raise ValueError('illegal state transition')
    if rights & required != required:
        raise PermissionError('insufficient rights')
    next_revision = order.revision + 1
    entry = AuditEntry28(order.phase, target, actor, next_revision)
    return OrderRecord28(order.order_id, target, next_revision, order.audit + (entry,))

original28 = OrderRecord28('ORD-1', OrderPhase28.NEW)
paid28 = change_order28(original28, target=OrderPhase28.PAID,
                        rights=OrderRight28.PAY, actor='cashier', expected_revision=0)
shipped28 = change_order28(paid28, target=OrderPhase28.SHIPPED,
                           rights=OrderRight28.SHIP, actor='warehouse', expected_revision=1)
print('Order history:', [(e.from_phase.value, e.to_phase.value) for e in shipped28.audit])

Order history: [('new', 'paid'), ('paid', 'shipped')]


**Capstone implementation, continued — JSON snapshot and restoration.**

In [57]:
def snapshot28(order: OrderRecord28) -> str:
    if not isinstance(order, OrderRecord28):
        raise TypeError('order required')
    doc = {
        'order_id': order.order_id,
        'phase': order.phase.value,
        'revision': order.revision,
        'audit': [
            {'from': e.from_phase.value, 'to': e.to_phase.value,
             'actor': e.actor, 'revision': e.revision}
            for e in order.audit
        ],
    }
    return json.dumps(doc, sort_keys=True)

def restore28(payload: str) -> OrderRecord28:
    doc = json.loads(payload)
    if not isinstance(doc, dict) or set(doc) != {'order_id', 'phase', 'revision', 'audit'}:
        raise ValueError('invalid order snapshot shape')
    if not isinstance(doc['audit'], list):
        raise ValueError('audit must be an array')
    audit = []
    for raw in doc['audit']:
        if not isinstance(raw, dict) or set(raw) != {'from', 'to', 'actor', 'revision'}:
            raise ValueError('invalid audit entry shape')
        audit.append(AuditEntry28(
            OrderPhase28(raw['from']), OrderPhase28(raw['to']),
            raw['actor'], raw['revision'],
        ))
    return OrderRecord28(doc['order_id'], OrderPhase28(doc['phase']),
                         doc['revision'], tuple(audit))

print('Canonical JSON:', snapshot28(shipped28))

Canonical JSON: {"audit": [{"actor": "cashier", "from": "new", "revision": 1, "to": "paid"}, {"actor": "warehouse", "from": "paid", "revision": 2, "to": "shipped"}], "order_id": "ORD-1", "phase": "shipped", "revision": 2}


**Verification — Problem 28.** These assertions cover ordinary inputs and failure modes.

In [58]:
assert original28.phase is OrderPhase28.NEW and original28.revision == 0
assert paid28.phase is OrderPhase28.PAID and paid28.revision == 1
assert shipped28.phase is OrderPhase28.SHIPPED and shipped28.revision == 2
assert len(shipped28.audit) == 2
assert restore28(snapshot28(shipped28)) == shipped28
assert restore28(snapshot28(original28)) == original28
with raises(RuntimeError):
    change_order28(paid28, target=OrderPhase28.SHIPPED, rights=OrderRight28.SHIP,
                   actor='warehouse', expected_revision=0)
with raises(PermissionError):
    change_order28(original28, target=OrderPhase28.PAID, rights=OrderRight28.CANCEL,
                   actor='stranger', expected_revision=0)
with raises(ValueError):
    change_order28(shipped28, target=OrderPhase28.CANCELLED, rights=OrderRight28.CANCEL,
                   actor='customer', expected_revision=2)
with raises(TypeError):
    change_order28(original28, target='paid', rights=OrderRight28.PAY,
                   actor='cashier', expected_revision=0)
with raises(ValueError): restore28('{"bad": true}')
with raises(ValueError):
    restore28(snapshot28(shipped28).replace('"revision": 2', '"revision": 99'))
assert original28 == OrderRecord28('ORD-1', OrderPhase28.NEW)
print("Problem 28: PASS")

Problem 28: PASS


**Stretch exercise (not required for the verified solution).** Harden `restore28` against non-string actor types, repeated JSON keys, and snapshot modification by a malicious client. For real systems, revision enforcement and audit persistence belong in one atomic database transaction; client-supplied audit trails need trusted integrity controls.

---
## Final review: ideas that transfer to production

- **Be explicit about representations.** Use `Enum[name]` for names and `Enum(value)` for stored values; use `.value` intentionally for stable JSON and database schemas.
- **Keep domain state out of members.** Enum members are shared singletons; represent per-entity state with distinct instances, immutable records, or database rows.
- **Make boundary validation strict.** Reject wrong types, unexpected names, unknown enum values, unsupported bitmasks, and inconsistent JSON shape.
- **Know the special cases.** Aliases appear in `__members__` but not ordinary iteration; `IntEnum`/`StrEnum` cross type boundaries; mutable `.value` payloads undermine deep immutability.
- **Test negative paths.** Each solution checks errors as well as success. For small flag domains, exhaustive tests are practical.
- **Plan migrations.** Treat enum wire values as compatibility-sensitive; provide explicit legacy decoders and canonical encoders.

**Independent challenge:** Implement an order workflow with a `RETURNED` state, a new `RETURN` permission, and an immutable audit record that includes a UTC timestamp supplied by a caller. State your expected behavior for repeated requests and rejected inputs, then write tests *before* coding.

In [59]:
# This cell runs only if all earlier assertions passed.
assert len(list(OrderPhase28)) == 4
assert checked25 == 64
assert restore28(snapshot28(shipped28)) == shipped28
print('ALL 28 PROBLEMS: SOLUTIONS AND ASSERTION TESTS COMPLETED SUCCESSFULLY')

ALL 28 PROBLEMS: SOLUTIONS AND ASSERTION TESTS COMPLETED SUCCESSFULLY
